import


In [79]:
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import seaborn as sns
import matplotlib.pyplot as plt



In [80]:
#load
data = pd.read_csv('Housing.csv')
data

,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished
...,...,...,...,...,...,...,...,...,...,...,...,...,...
540,1820000,3000,2,1,1,yes,no,yes,no,no,2,no,unfurnished
541,1767150,2400,3,1,1,no,no,no,no,no,0,no,semi-furnished
542,1750000,3620,2,1,1,yes,no,no,no,no,0,no,unfurnished
543,1750000,2910,3,1,1,no,no,no,no,no,0,no,furnished


In [81]:
#encode categorical cols
label_encoder = LabelEncoder()
categorical_columns =  [
    'mainroad', 'guestroom', 'basement', 'hotwaterheating',
    'airconditioning', 'prefarea', 'furnishingstatus'
]

for col in categorical_columns:
    data[col] = label_encoder.fit_transform(data[col])

In [82]:
#remove outliers
data = data[data['area'] < data['area'].quantile(0.95)]
data = data[data['area'] < data['area'].quantile(0.95)]

In [83]:
#split dataset into features(X) and target(y)
X = data.drop(columns = ['price'])
y = data['price']

In [84]:
#standardise / scale numerical features
scaler = StandardScaler()
numerical_cols = ['area', 'bedrooms' , 'bathrooms', 'stories', 'parking']
X[numerical_cols] = scaler.fit_transform(X[numerical_cols])


In [85]:
data.columns

Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'airconditioning',
       'parking', 'prefarea', 'furnishingstatus'],
      dtype='object')

In [86]:
#split into train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [87]:
from sklearn.linear_model import Ridge

#intialize and train ml model
# linear_model = LinearRegression()
# linear_model.fit(X_train, y_train)
param_grid = {'alpha': [0.1, 1, 10, 100, 1000]}
ridge = Ridge()
grid_search = GridSearchCV(ridge, param_grid, scoring='neg_mean_squared_error', cv=5)
grid_search.fit(X_train, y_train)

best_alpha = grid_search.best_params_['alpha']

best_model = grid_search.best_estimator_

In [88]:
#func to take in inputs and predict the house price
def enter_details():
    print("Enter the details of the house: ")
    area = float(input("Enter the area of the house: "))
    bedrooms = int(input("Enter the bedrooms of the house: "))
    bathrooms = int(input("Enter the bathrooms of the house: "))
    stories = int(input("enter the number of stories: "))
    mainroad = int(input("enter 0 for no access and 1 for access to main road: "))
    guestroom = int(input("Enter the number of guestrooms: "))
    basement = int(input("Enter the number of basements: "))
    hotwaterheating = int(input("Enter 0 or 1 for access to hotwaterheating: "))
    airconditioning = int(input("Enter 0 or 1 for access to airconditioning: "))
    parking = int(input("Enter 0 or 1 for access to parking: "))
    prefarea = int(input("Enter 0 or 1 for access to prefarea: "))
    furnishingstatus = int(input("Enter 0 or 1 or 2 for access to furnishingstatus: "))

    #creating a dataframe for the input
    input_data = pd.DataFrame([[area, bedrooms, bathrooms, stories, mainroad, guestroom, basement, hotwaterheating, airconditioning, parking, prefarea, furnishingstatus ]], columns = ['area', 'bedrooms', 'bathrooms', 'stories', 'mainroad', 'guestroom', 'basement', 'hotwaterheating', 'airconditioning','parking', 'prefarea', 'furnishingstatus'])

    return input_data

    #predict the price
    # print(input_data[0])
def predict_house_price(input_data):
    predicted_price = linear_model.predict(input_data)[0]
    print("the predicted price is: ", predicted_price)

In [89]:
# #calling the predict func
# input_det = enter_details()
# predict_house_price(input_det)

In [90]:
from sklearn.decomposition import PCA

pca = PCA(n_components=10)  # Adjust components based on variance explained
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

ridge.fit(X_train_pca, y_train)
y_pred_pca = ridge.predict(X_test_pca)

mse_pca = mean_squared_error(y_test, y_pred_pca)
r2_pca = r2_score(y_test, y_pred_pca)
print(f"PCA Adjusted Ridge Regression MSE: {mse_pca}")
print(f"PCA Adjusted Ridge Regression R^2 Score: {r2_pca}")

PCA Adjusted Ridge Regression MSE: 1128575521970.475
PCA Adjusted Ridge Regression R^2 Score: 0.7075000205316679


In [91]:
#evaluate the model
# y_pred = linear_model.predict(X_test)
y_pred = best_model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("mse: ", mse)
print("r2: ", r2)

mse:  1115837444124.8618
r2:  0.7108014278684163
